In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
import joblib
import threading
import subprocess

from math import sqrt
from collections import deque

# ==============================
# LOAD MODEL
# ==============================
model = joblib.load("driver_model.pkl")

# ==============================
# MEDIAPIPE SETUP
# ==============================
mp_face_mesh = mp.solutions.face_mesh
mp_hands = mp.solutions.hands

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    refine_landmarks=True,
    max_num_faces=1
)

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1
)

# ==============================
# LANDMARK INDICES
# ==============================
LEFT_EYE  = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
LEFT_IRIS = [468, 469, 470, 471]
RIGHT_IRIS = [472, 473, 474, 475]

NOSE_TIP = 1

# ==============================
# THRESHOLDS
# ==============================
EYE_CLOSED_THRESH = 0.20

HEAD_PITCH_DOWN_THRESH = -15
GAZE_Y_DOWN_THRESH = 6

UNSAFE_TIME_THRESHOLD = 3.0

RISK_WARNING = 0.45
RISK_ALERT = 0.55

RISK_WINDOW = 30
risk_buffer = deque(maxlen=RISK_WINDOW)

SIDE_DIST_TIME_THRESHOLD = 4.0

HIGH_RISK_THRESHOLD = 0.65
HIGH_RISK_TIME_THRESHOLD = 4.0

# ==============================
# STATES
# ==============================
unsafe_start_time = None
side_start_time = None
high_risk_start_time = None
buzzer_playing = False

# ==============================
# UTILS
# ==============================
def euclidean(p1, p2):
    return sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def eye_aspect_ratio(eye):
    A = euclidean(eye[1], eye[5])
    B = euclidean(eye[2], eye[4])
    C = euclidean(eye[0], eye[3])
    return (A + B) / (2.0 * C)

def play_buzzer():
    global buzzer_playing
    buzzer_playing = True
    for _ in range(4):
        subprocess.call(
            ["afplay", "/System/Library/Sounds/Glass.aiff"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
    buzzer_playing = False

def get_head_pose(image, face):
    h, w, _ = image.shape

    image_points = np.array([
        (face[1].x * w, face[1].y * h),
        (face[152].x * w, face[152].y * h),
        (face[33].x * w, face[33].y * h),
        (face[263].x * w, face[263].y * h),
        (face[61].x * w, face[61].y * h),
        (face[291].x * w, face[291].y * h)
    ], dtype="double")

    model_points = np.array([
        (0, 0, 0),
        (0, -63.6, -12.5),
        (-43.3, 32.7, -26.0),
        (43.3, 32.7, -26.0),
        (-28.9, -28.9, -24.1),
        (28.9, -28.9, -24.1)
    ])

    focal_length = w
    center = (w / 2, h / 2)
    camera_matrix = np.array([
        [focal_length, 0, center[0]],
        [0, focal_length, center[1]],
        [0, 0, 1]
    ])

    _, rvec, _ = cv2.solvePnP(
        model_points, image_points, camera_matrix, np.zeros((4, 1))
    )

    rmat, _ = cv2.Rodrigues(rvec)
    angles, _, _, _, _, _ = cv2.RQDecomp3x3(rmat)

    return angles[0]*360, angles[1]*360, angles[2]*360

# ==============================
# VIDEO SOURCE
# ==============================
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Video not opened")

print("Driver Monitoring Started (Press Q to quit)")

# ==============================
# MAIN LOOP
# ==============================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    display = frame.copy()
    h, w, _ = frame.shape
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_results = face_mesh.process(rgb)
    hand_results = hands.process(rgb)

    status = "NO FACE"
    color = (0, 255, 255)

    if face_results.multi_face_landmarks:
        face = face_results.multi_face_landmarks[0].landmark
        def pt(i): return np.array([face[i].x * w, face[i].y * h])

        # Face box
        xs = [lm.x for lm in face]
        ys = [lm.y for lm in face]
        x_min, x_max = int(min(xs)*w), int(max(xs)*w)
        y_min, y_max = int(min(ys)*h), int(max(ys)*h)

        # EAR
        left_eye = [pt(i) for i in LEFT_EYE]
        right_eye = [pt(i) for i in RIGHT_EYE]
        ear = (eye_aspect_ratio(left_eye) + eye_aspect_ratio(right_eye)) / 2

        eye_state = 0 if ear < 0.18 else 1 if ear < 0.23 else 2

        # Gaze Y
        gaze_y = (
            np.mean([pt(i)[1] for i in LEFT_IRIS + RIGHT_IRIS]) -
            np.mean([pt(i)[1] for i in LEFT_EYE + RIGHT_EYE])
        )

        # Head pose
        pitch, yaw, roll = get_head_pose(frame, face)

        # Rule-based distraction
        rule_distracted = False
        if abs(yaw) < 20 and pitch < HEAD_PITCH_DOWN_THRESH and gaze_y > GAZE_Y_DOWN_THRESH:
            rule_distracted = True

        side_distracted = abs(yaw) > 30 and abs(pitch) < 10

        # Hand distance
        hand_face_dist = 0
        if hand_results.multi_hand_landmarks:
            hand = hand_results.multi_hand_landmarks[0].landmark
            hand_face_dist = euclidean(
                np.array([hand[8].x*w, hand[8].y*h]),
                pt(NOSE_TIP)
            )

        # ==============================
        # ML RISK
        # ==============================
        features = np.array([[ear, eye_state, 0, gaze_y, pitch, yaw, roll, hand_face_dist]])
        risk = model.predict_proba(features)[0][1]

        risk_buffer.append(risk)
        smoothed_risk = np.mean(risk_buffer)

        current_time = time.time()
        status = "SAFE"
        color = (0, 255, 0)

        # ==============================
        # HIGH RISK OVERRIDE
        # ==============================
        if smoothed_risk >= HIGH_RISK_THRESHOLD:
            if high_risk_start_time is None:
                high_risk_start_time = current_time

            if (current_time - high_risk_start_time) >= HIGH_RISK_TIME_THRESHOLD:
                status = "DISTRACTED (HIGH RISK)"
                color = (0, 0, 255)
                if not buzzer_playing:
                    threading.Thread(target=play_buzzer, daemon=True).start()
        else:
            high_risk_start_time = None

        # ==============================
        # NORMAL LOGIC
        # ==============================
        if status == "SAFE":
            if rule_distracted and smoothed_risk >= RISK_ALERT:
                if unsafe_start_time is None:
                    unsafe_start_time = current_time

                if (current_time - unsafe_start_time) >= UNSAFE_TIME_THRESHOLD:
                    status = "DISTRACTED"
                    color = (0, 0, 255)
                    if not buzzer_playing:
                        threading.Thread(target=play_buzzer, daemon=True).start()

            elif smoothed_risk >= RISK_WARNING:
                status = "WARNING"
                color = (0, 255, 255)
                unsafe_start_time = None
            else:
                unsafe_start_time = None

        # ==============================
        # SIDE DISTRACTION
        # ==============================
        if status == "SAFE" and side_distracted:
            if side_start_time is None:
                side_start_time = current_time

            if (current_time - side_start_time) >= SIDE_DIST_TIME_THRESHOLD:
                status = "DISTRACTED (SIDE)"
                color = (0, 165, 255)
                if not buzzer_playing:
                    threading.Thread(target=play_buzzer, daemon=True).start()
        else:
            side_start_time = None

        # ==============================
        # VISUALS
        # ==============================
        cv2.rectangle(display, (x_min, y_min), (x_max, y_max), color, 4)
        cv2.putText(display, status, (x_min, y_min - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)

        cv2.putText(display, f"Risk: {smoothed_risk:.2f}",
                    (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)

    cv2.imshow("Driver Monitoring", display)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


I0000 00:00:1769228143.161479       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
I0000 00:00:1769228143.168087       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Driver Monitoring Started (Press Q to quit)


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.